## 1. Import Libraries

In [1]:
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from golf_pipeline.video_io import extract_frames_from_video
from golf_pipeline.preprocessing import StylePreprocessor

# Setup matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Configuration

In [12]:
# ==================== CONFIGURATION ====================
# Input dataset path
AUGMENTED_DATASET_PATH = Path("./augmented_dataset/Trong nhà - Indoor")

# Output directory
OUTPUT_DIR = Path("preprocessed_dataset")
OUTPUT_VIDEO_DIR = OUTPUT_DIR / "videos"

# Create output directories
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_VIDEO_DIR.mkdir(exist_ok=True)

# Preprocessing configuration - Tắt các bước chậm để xử lý nhanh hơn
PREPROCESSING_CONFIG = {
    "use_adaptive": False,           # Tắt - chậm
    "apply_sharpening": False,       # Tắt - không cần thiết
    "sharpen_strength": 0.8,
    "apply_edge_enhancement": False, # Tắt - chậm
    "edge_alpha": 0.3,
    "apply_denoise": False,          # Tắt - RẤT CHẬM!
    "gamma": 1.0,                    # Không thay đổi
    "contrast_alpha": 1.0,           # Không thay đổi
    "contrast_beta": 0,              # Không thay đổi
}

# Video processing
FRAME_SKIP = 1  # Process every frame
MAX_FRAMES = None  # None = process entire video

# Video encoding settings
VIDEO_CODEC = 'mp4v'  # Codec for MP4 output

print(f"Input dataset: {AUGMENTED_DATASET_PATH}")
print(f"Output directory: {OUTPUT_VIDEO_DIR}")
print(f"⚡ Fast mode: Preprocessing disabled for speed")

Input dataset: augmented_dataset\Trong nhà - Indoor
Output directory: preprocessed_dataset\videos
⚡ Fast mode: Preprocessing disabled for speed


## 3. Scan Dataset

In [13]:
# Find all video files in the augmented dataset
video_files = []

# for environment in ["Ngoài trời - Outdoor", "Trong nhà - Indoor"]:
env_path = AUGMENTED_DATASET_PATH 
#     if not env_path.exists():
#         print(f"Warning: {environment} directory not found")
#         continue
    
#     # Find all band directories
for band_dir in sorted(env_path.iterdir()):
    if not band_dir.is_dir():
        continue
    
    # Find all video files in this band
    for video_file in sorted(band_dir.glob("*.mp4")):
        video_files.append({
            "path": video_file,
            "environment": environment.split(" - ")[1],  # Outdoor or Indoor
            "band": band_dir.name,
            "filename": video_file.name
        })

print(f"✓ Found {len(video_files)} videos to process")
print(f"\nBreakdown by environment:")
for env in ["Outdoor", "Indoor"]:
    count = sum(1 for v in video_files if v['environment'] == env)
    print(f"  {env}: {count} videos")

# Display sample
print("\nSample videos:")
for video_info in video_files[:5]:
    print(f"  {video_info['environment']} / {video_info['band']} / {video_info['filename']}")

✓ Found 104 videos to process

Breakdown by environment:
  Outdoor: 0 videos
  Indoor: 104 videos

Sample videos:
  Indoor / Band 1-2 / Backside-8767-14_no00.mp4
  Indoor / Band 1-2 / Backside-8767-14_no01.mp4
  Indoor / Band 1-2 / Backside-8767-14_no02.mp4
  Indoor / Band 1-2 / Backside-8767-14_no03.mp4
  Indoor / Band 1-2 / Backside-8767-16_no00.mp4


## 4. Process Videos

For each video:
1. Extract frames
2. Apply preprocessing
3. Save preprocessed video

In [14]:
def preprocess_frame(frame, preprocessor, config):
    """Apply preprocessing steps to a frame."""
    processed = frame.copy()
    
    # Apply style normalization
    if config.get('use_adaptive', False):
        processed = preprocessor.match_color_style(processed)
    else:
        processed = preprocessor.normalize_basic(processed)
    
    # Apply sharpening
    if config.get('apply_sharpening', False):
        kernel = np.array([[-1, -1, -1],
                          [-1,  9, -1],
                          [-1, -1, -1]]) * config.get('sharpen_strength', 0.8)
        processed = cv2.filter2D(processed, -1, kernel)
    
    # Apply edge enhancement
    if config.get('apply_edge_enhancement', False):
        edges = cv2.Canny(cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY), 50, 150)
        edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
        alpha = config.get('edge_alpha', 0.3)
        processed = cv2.addWeighted(processed, 1, edges_colored, alpha, 0)
    
    # Apply denoising
    if config.get('apply_denoise', False):
        processed = cv2.fastNlMeansDenoisingColored(processed, None, 10, 10, 7, 21)
    
    # Apply gamma correction
    gamma = config.get('gamma', 1.0)
    if gamma != 1.0:
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype("uint8")
        processed = cv2.LUT(processed, table)
    
    # Apply contrast adjustment
    alpha = config.get('contrast_alpha', 1.0)
    beta = config.get('contrast_beta', 0)
    if alpha != 1.0 or beta != 0:
        processed = cv2.convertScaleAbs(processed, alpha=alpha, beta=beta)
    
    return processed


def process_and_save_video(video_path, output_path, preprocessor, config):
    """Process a single video and save the preprocessed version."""
    # Load video
    frames, fps = extract_frames_from_video(
        str(video_path),
        sample_every=FRAME_SKIP,
        max_frames=MAX_FRAMES
    )
    
    if len(frames) == 0:
        return 0, fps
    
    # Get frame dimensions
    height, width = frames[0].shape[:2]
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*VIDEO_CODEC)
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
    
    # Preprocess and write frames
    for frame in frames:
        processed = preprocess_frame(frame, preprocessor, config)
        out.write(processed)
    
    # Release video writer
    out.release()
    
    return len(frames), fps


print("Starting video processing...\n")

Starting video processing...



In [15]:
# Process all videos
processing_stats = []

for video_info in tqdm(video_files, desc="Processing videos"):
    video_path = video_info['path']
    
    # Create output filename (keep same structure)
    output_filename = f"{video_info['environment']}_{video_info['band']}_{video_info['filename']}"
    output_path = OUTPUT_VIDEO_DIR / output_filename
    
    # Skip if already processed
    if output_path.exists():
        print(f"  Skipping {video_info['filename']} (already processed)")
        continue
    
    try:
        # Initialize preprocessor with first frame
        frames, fps = extract_frames_from_video(str(video_path), sample_every=FRAME_SKIP, max_frames=1)
        preprocessor = StylePreprocessor(reference_image=frames[0])
        
        # Process and save video
        num_frames, fps = process_and_save_video(
            video_path,
            output_path,
            preprocessor,
            PREPROCESSING_CONFIG
        )
        
        # Record stats
        processing_stats.append({
            'video': video_info['filename'],
            'environment': video_info['environment'],
            'band': video_info['band'],
            'frames_processed': num_frames,
            'fps': fps,
            'status': 'success'
        })
        
    except Exception as e:
        print(f"  Error processing {video_info['filename']}: {e}")
        processing_stats.append({
            'video': video_info['filename'],
            'environment': video_info['environment'],
            'band': video_info['band'],
            'frames_processed': 0,
            'fps': 0,
            'status': f'error: {str(e)}'
        })

print("\n✓ Processing completed!")

Processing videos:   0%|          | 0/104 [00:00<?, ?it/s]

  Skipping Backside-8767-14_no00.mp4 (already processed)
  Skipping Backside-8767-14_no01.mp4 (already processed)
  Skipping Backside-8767-14_no02.mp4 (already processed)
  Skipping Backside-8767-14_no03.mp4 (already processed)
  Skipping Backside-8767-16_no00.mp4 (already processed)
  Skipping Backside-8767-16_no01.mp4 (already processed)
  Skipping Backside-8767-16_no02.mp4 (already processed)
  Skipping Backside-8767-16_no03.mp4 (already processed)
  Skipping Backside-8767-17_no00.mp4 (already processed)
  Skipping Backside-8767-17_no01.mp4 (already processed)
  Skipping Backside-8767-17_no02.mp4 (already processed)
  Skipping Backside-8767-17_no03.mp4 (already processed)
  Skipping Side-6020-15_no00.mp4 (already processed)
  Skipping Side-6020-15_no01.mp4 (already processed)
  Skipping Side-6020-15_no02.mp4 (already processed)
  Skipping Side-6020-15_no03.mp4 (already processed)
  Skipping Side-6020-7_no00.mp4 (already processed)
  Skipping Side-6020-7_no01.mp4 (already processed)


## 5. Summary Statistics

In [ ]:
# Create summary dataframe
stats_df = pd.DataFrame(processing_stats)

print("=" * 80)
print("PROCESSING SUMMARY")
print("=" * 80)

# Overall stats
total_videos = len(stats_df)
successful = len(stats_df[stats_df['status'] == 'success'])
failed = total_videos - successful

print(f"\nTotal videos processed: {total_videos}")
print(f"  ✓ Successful: {successful}")
print(f"  ✗ Failed: {failed}")

# Environment breakdown
print("\nBreakdown by environment:")
for env in stats_df['environment'].unique():
    env_df = stats_df[stats_df['environment'] == env]
    success_count = len(env_df[env_df['status'] == 'success'])
    print(f"  {env}: {success_count}/{len(env_df)} successful")

# Band breakdown
print("\nBreakdown by band:")
for band in sorted(stats_df['band'].unique()):
    band_df = stats_df[stats_df['band'] == band]
    success_count = len(band_df[band_df['status'] == 'success'])
    print(f"  {band}: {success_count}/{len(band_df)} successful")

# Total frames
total_frames = stats_df[stats_df['status'] == 'success']['frames_processed'].sum()
print(f"\nTotal frames processed: {total_frames:,}")

# Output files
output_files = list(OUTPUT_VIDEO_DIR.glob("*.mp4"))
print(f"\nOutput videos created: {len(output_files)}")
print(f"Output directory: {OUTPUT_VIDEO_DIR}")

# Calculate total output size
total_size_mb = sum(f.stat().st_size for f in output_files) / (1024 * 1024)
print(f"Total output size: {total_size_mb:.2f} MB")

# Show errors if any
if failed > 0:
    print("\nFailed videos:")
    for _, row in stats_df[stats_df['status'] != 'success'].iterrows():
        print(f"  {row['video']}: {row['status']}")

print("\n" + "=" * 80)

# Save summary
summary_path = OUTPUT_DIR / "processing_summary.csv"
stats_df.to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to: {summary_path}")

## 6. Sample Visualization

In [ ]:
# Load and visualize a sample video
output_files = list(OUTPUT_VIDEO_DIR.glob("*.mp4"))

if len(output_files) > 0:
    sample_file = output_files[0]
    print(f"Sample video: {sample_file.name}")
    
    # Get video info
    cap = cv2.VideoCapture(str(sample_file))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    print(f"\nVideo info:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps:.2f}")
    print(f"  Frames: {frame_count}")
    print(f"  Duration: {duration:.2f} seconds")
    print(f"  Size: {sample_file.stat().st_size / (1024*1024):.2f} MB")
    
    # Extract and display sample frames
    sample_indices = [0, frame_count//4, frame_count//2, max(0, frame_count-1)]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for ax, idx in zip(axes, sample_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Frame {idx} ({idx/fps:.2f}s)")
            ax.axis('off')
    
    cap.release()
    plt.tight_layout()
    plt.show()
    
    # Display video in notebook
    print("\nVideo player:")
    display(Video(str(sample_file), embed=True, width=800))
else:
    print("No output videos found.")